In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

SILVER_CORE_TABLE = f"{CATALOG}.{SCHEMA}.silver_events_core"
ACTOR_IDENTITY_TABLE = f"{CATALOG}.{SCHEMA}.dim_actor_identity"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_repo_daily_activity"

In [0]:
required_tables = [
    "workspace.oss_ecosystem.gold_repo_daily_activity",
    "workspace.oss_ecosystem.silver_events_core",
    "workspace.oss_ecosystem.dim_actor_identity"
]

for table_name in required_tables:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(f"Required table missing: {table_name}")

print("All required lifecycle input tables exist.")

In [0]:
human_repo_activity_df = (
    spark.table(SILVER_CORE_TABLE)
    .filter(
        F.col("repo_id").isNotNull() &
        F.col("actor_id").isNotNull()
    )
    .join(
        spark.table(ACTOR_IDENTITY_TABLE)
        .select("actor_id", "actor_type"),
        on="actor_id",
        how="inner"
    )
    .filter(
        F.col("actor_type") == "human_or_unknown"
    )
    .withColumn(
        "event_date",
        F.to_date("event_ts")
    )
    .select(
        "repo_id",
        "actor_id",
        "event_date"
    )
    .distinct()
)

In [0]:
repo_actor_first_seen_df = (
    human_repo_activity_df
    .groupBy(
        "repo_id",
        "actor_id"
    )
    .agg(
        F.min("event_date").alias("first_observed_date")
    )
)

In [0]:
contributor_lifecycle_df = (
    human_repo_activity_df
    .join(
        repo_actor_first_seen_df,
        on=["repo_id", "actor_id"],
        how="inner"
    )
    .withColumn(
        "contributor_status",
        F.when(
            F.col("event_date") == F.col("first_observed_date"),
            F.lit("newly_observed")
        ).otherwise(
            F.lit("returning")
        )
    )
)

In [0]:
contributor_lifecycle_df.groupBy(
    "contributor_status"
).agg(
    F.count("*").alias("repo_day_contributor_occurrences"),
    F.countDistinct("actor_id").alias("distinct_actors")
).show()

In [0]:
repo_daily_lifecycle_df = (
    contributor_lifecycle_df
    .groupBy(
        "repo_id",
        "event_date"
    )
    .agg(
        F.countDistinct(
            F.when(
                F.col("contributor_status") == "newly_observed",
                F.col("actor_id")
            )
        ).alias("newly_observed_contributors"),

        F.countDistinct(
            F.when(
                F.col("contributor_status") == "returning",
                F.col("actor_id")
            )
        ).alias("returning_contributors")
    )
)

In [0]:
gold_with_lifecycle_df = (
    spark.table(GOLD_TABLE)
    .join(
        repo_daily_lifecycle_df,
        on=["repo_id", "event_date"],
        how="left"
    )
    .fillna(
        0,
        subset=[
            "newly_observed_contributors",
            "returning_contributors"
        ]
    )
)

In [0]:
print(
    "Gold rows:",
    gold_with_lifecycle_df.count()
)

duplicate_gold_keys = (
    gold_with_lifecycle_df
    .groupBy("repo_id", "event_date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate repo/date keys:",
    duplicate_gold_keys
)

gold_with_lifecycle_df.select(
    F.sum("newly_observed_contributors")
        .alias("newly_observed_repo_day_total"),

    F.sum("returning_contributors")
        .alias("returning_repo_day_total")
).show()

In [0]:
(
    gold_with_lifecycle_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

In [0]:
final_gold_df = spark.table(GOLD_TABLE)

print(
    "Final Gold rows:",
    final_gold_df.count()
)

print(
    "Final duplicate repo/date keys:",
    final_gold_df
    .groupBy("repo_id", "event_date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

final_gold_df.select(
    F.sum("newly_observed_contributors")
        .alias("newly_observed_total"),

    F.sum("returning_contributors")
        .alias("returning_total")
).show()